In [1]:
# =========================================
# Cell 1 — Install dependencies
# =========================================
!pip -q install openai>=1.0.0 datasets>=2.19.0 pandas>=2.2.0 numpy>=1.26.0 scikit-learn>=1.3.0 tqdm>=4.66.0 matplotlib>=3.8.0


In [2]:
# =========================================
# Cell 2 — Keys (type/paste in)
# =========================================
import os, getpass

os.environ["DEEPSEEK_API_KEY"] = getpass.getpass("Enter DEEPSEEK_API_KEY (hidden): ")
os.environ["HF_TOKEN"] = getpass.getpass("Enter HF_TOKEN (required for gated flare-fiqasa): ").strip()

assert os.environ.get("DEEPSEEK_API_KEY"), "Missing DEEPSEEK_API_KEY"
assert os.environ.get("HF_TOKEN"), "Missing HF_TOKEN"
print("✅ Keys set.")


Enter DEEPSEEK_API_KEY (hidden): ··········
Enter HF_TOKEN (required for gated flare-fiqasa): ··········
✅ Keys set.


In [3]:
# =========================================
# Cell 3 — Load FIQASA dataset (flare-fiqasa) from Hugging Face
# =========================================
from datasets import load_dataset

DATASET_NAME = "TheFinAI/flare-fiqasa"
SPLIT = "test"   # "train", "valid", or "test"

ds = load_dataset(DATASET_NAME, split=SPLIT, token=os.environ["HF_TOKEN"])
print(ds)
print("Columns:", ds.column_names)
print("Example row:\n", ds[0])


README.md:   0%|          | 0.00/1.13k [00:00<?, ?B/s]

data/train-00000-of-00001-d0f9b6513e12e0(…):   0%|          | 0.00/100k [00:00<?, ?B/s]

data/test-00000-of-00001-faca082021057ac(…):   0%|          | 0.00/35.8k [00:00<?, ?B/s]

data/valid-00000-of-00001-36997935dc03cb(…):   0%|          | 0.00/29.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/750 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/235 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/188 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'query', 'answer', 'text', 'choices', 'gold'],
    num_rows: 235
})
Columns: ['id', 'query', 'answer', 'text', 'choices', 'gold']
Example row:
 {'id': 'fiqasa938', 'query': 'What is the sentiment of the following financial post: Positive, Negative, or Neutral?\nText: $BBRY Actually lost .03c per share if U incl VZ as no debt and 3.1 in Cash.\nAnswer:', 'answer': 'negative', 'text': '$BBRY Actually lost .03c per share if U incl VZ as no debt and 3.1 in Cash.', 'choices': ['negative', 'positive', 'neutral'], 'gold': 0}


In [4]:
# =========================================
# Cell 4 — DeepSeek Thinking (deepseek-reasoner): prompt + API helpers
# =========================================
import re, time
from openai import OpenAI

DEEPSEEK_BASE_URL = "https://api.deepseek.com"  # OpenAI-compatible base_url :contentReference[oaicite:3]{index=3}
MODEL_ID = "deepseek-reasoner"                  # thinking/reasoning model :contentReference[oaicite:4]{index=4}

client = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url=DEEPSEEK_BASE_URL
)

def build_prompt(text: str, choices):
    return (
        "Analyze the sentiment of this financial text.\n"
        "Respond ONLY with one label from the Options.\n\n"
        f"Text: {text}\n"
        f"Options: {', '.join(map(str, choices))}\n"
        "Label:"
    )

def extract_label(model_text: str, choices):
    if not model_text:
        return None
    s = model_text.strip().lower()

    for c in choices:  # exact match
        if s == str(c).strip().lower():
            return str(c)

    for c in choices:  # whole-word match (handles extra words)
        pat = r"\b" + re.escape(str(c).strip().lower()) + r"\b"
        if re.search(pat, s):
            return str(c)

    return None

def deepseek_reasoner_predict(prompt: str, model: str = MODEL_ID, max_retries: int = 6, sleep_base: float = 1.0):
    last_err = None
    for attempt in range(max_retries):
        try:
            # OpenAI-compatible Chat Completions format :contentReference[oaicite:5]{index=5}
            resp = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": "You are a financial analyst."},
                    {"role": "user", "content": prompt},
                ],
                temperature=0.0,
                stream=False,
            )

            text_out = resp.choices[0].message.content if resp and resp.choices else None

            usage = getattr(resp, "usage", None)
            usage_dict = None
            if usage is not None:
                usage_dict = {
                    "prompt_tokens": getattr(usage, "prompt_tokens", None),
                    "completion_tokens": getattr(usage, "completion_tokens", None),
                    "total_tokens": getattr(usage, "total_tokens", None),
                }

            meta = {"usage": usage_dict}
            return text_out, meta

        except Exception as e:
            last_err = str(e)
            time.sleep(sleep_base * (2 ** attempt))

    return None, {"error": last_err}


In [5]:
# =========================================
# Cell 5 — Run evaluation (full split by default) + resume support
# =========================================
import os, json, random, time
from tqdm import tqdm

SEED = 42
MAX_SAMPLES = None         # None = FULL split; set int for quick test
SHUFFLE = True
SLEEP_BETWEEN_CALLS = 0.3  # raise to 0.5–1.0 if rate-limited

random.seed(SEED)

os.makedirs("text_responses", exist_ok=True)
os.makedirs("evaluation", exist_ok=True)

run_tag = f"deepseek_thinking_{MODEL_ID}_{DATASET_NAME.replace('/','_')}_{SPLIT}"
raw_path = f"text_responses/{run_tag}.jsonl"

# resume
done = set()
if os.path.exists(raw_path):
    with open(raw_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                done.add(json.loads(line)["row_index"])
            except:
                pass
print("Already completed:", len(done))

idxs = list(range(len(ds)))
if SHUFFLE:
    random.shuffle(idxs)
if MAX_SAMPLES is not None:
    idxs = idxs[:MAX_SAMPLES]

with open(raw_path, "a", encoding="utf-8") as f:
    for i in tqdm(idxs, desc="Evaluating"):
        if i in done:
            continue

        ex = ds[i]
        text = ex["text"]
        choices = ex["choices"]
        gold = int(ex["gold"])

        prompt = build_prompt(text, choices)
        model_out, meta = deepseek_reasoner_predict(prompt)

        pred_label = extract_label(model_out, choices)
        pred_idx = choices.index(pred_label) if pred_label in choices else -1

        rec = {
            "row_index": i,
            "id": ex.get("id", None),
            "query": ex.get("query", None),
            "answer": ex.get("answer", None),
            "text": text,
            "choices": choices,
            "gold": gold,
            "model_response_raw": model_out,
            "predicted_label": pred_label,
            "predicted_index": pred_idx,
            "correct": (pred_idx == gold),
            "meta": meta,  # JSON-serializable
        }

        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

        if SLEEP_BETWEEN_CALLS:
            time.sleep(SLEEP_BETWEEN_CALLS)

print("✅ Saved raw responses to:", raw_path)


Already completed: 0


Evaluating: 100%|██████████| 235/235 [1:02:21<00:00, 15.92s/it]

✅ Saved raw responses to: text_responses/deepseek_thinking_deepseek-reasoner_TheFinAI_flare-fiqasa_test.jsonl


In [6]:
# =========================================
# Cell 6 — Compute metrics + save artifacts (loads full jsonl)
# =========================================
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import json

all_rows = []
with open(raw_path, "r", encoding="utf-8") as f:
    for line in f:
        try:
            all_rows.append(json.loads(line))
        except:
            pass

df = pd.DataFrame(all_rows)

valid_mask = df["predicted_index"] >= 0
labels_sorted = sorted(set(df["gold"].tolist()))

acc_all = accuracy_score(df["gold"], df["predicted_index"])
acc_valid = accuracy_score(df.loc[valid_mask, "gold"], df.loc[valid_mask, "predicted_index"]) if valid_mask.any() else None

metrics = {
    "run_tag": run_tag,
    "model_id": MODEL_ID,
    "dataset": DATASET_NAME,
    "split": SPLIT,
    "num_samples": int(len(df)),
    "num_valid_predictions": int(valid_mask.sum()),
    "accuracy_all": float(acc_all),
    "accuracy_valid_only": (float(acc_valid) if acc_valid is not None else None),
    "f1_macro_valid_only": float(f1_score(df.loc[valid_mask, "gold"], df.loc[valid_mask, "predicted_index"],
                                         average="macro", labels=labels_sorted)) if valid_mask.any() else None,
    "f1_micro_valid_only": float(f1_score(df.loc[valid_mask, "gold"], df.loc[valid_mask, "predicted_index"],
                                         average="micro", labels=labels_sorted)) if valid_mask.any() else None,
    "f1_weighted_valid_only": float(f1_score(df.loc[valid_mask, "gold"], df.loc[valid_mask, "predicted_index"],
                                            average="weighted", labels=labels_sorted)) if valid_mask.any() else None,
}

cm = confusion_matrix(df.loc[valid_mask, "gold"], df.loc[valid_mask, "predicted_index"], labels=labels_sorted) if valid_mask.any() else None

metrics_path = f"evaluation/{run_tag}_metrics.json"
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(
        {"metrics": metrics, "confusion_matrix_labels": labels_sorted, "confusion_matrix": (cm.tolist() if cm is not None else None)},
        f, ensure_ascii=False, indent=2
    )

preds_path = f"evaluation/{run_tag}_predictions.csv"
df.to_csv(preds_path, index=False)

print("✅ Metrics saved to:", metrics_path)
print("✅ Predictions saved to:", preds_path)

if valid_mask.any():
    print("\nClassification report (valid predictions only):")
    print(classification_report(df.loc[valid_mask, "gold"], df.loc[valid_mask, "predicted_index"], labels=labels_sorted))
else:
    print("No valid predictions to report.")


✅ Metrics saved to: evaluation/deepseek_thinking_deepseek-reasoner_TheFinAI_flare-fiqasa_test_metrics.json
✅ Predictions saved to: evaluation/deepseek_thinking_deepseek-reasoner_TheFinAI_flare-fiqasa_test_predictions.csv

Classification report (valid predictions only):
              precision    recall  f1-score   support

           0       0.88      0.86      0.87        76
           1       0.95      0.77      0.85       141
           2       0.26      0.67      0.38        18

    accuracy                           0.79       235
   macro avg       0.70      0.76      0.70       235
weighted avg       0.87      0.79      0.82       235

